## 1: Start Spark with the JDBC jar

In [1]:
from pyspark.sql import SparkSession
import os

jdbc_jar = os.path.expanduser("~/bigdata_lab/jars/mssql-jdbc.jar")

spark = SparkSession.builder \
    .appName("PersonA-Sources") \
    .config("spark.jars", jdbc_jar) \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/31 15:08:14 WARN Utils: Your hostname, Go, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/31 15:08:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/07/31 15:08:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/ajith/bigdata_lab/venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


## 2: Load .env credentials

In [2]:
from dotenv import load_dotenv
load_dotenv()

AZURE_JDBC_URL = (
    f"jdbc:sqlserver://{os.environ['AZURE_SQL_SERVER']}:1433;"
    f"database={os.environ['AZURE_SQL_DATABASE']};"
    "encrypt=true;trustServerCertificate=false;"
)
AZURE_USER = os.environ["AZURE_SQL_USERNAME"]
AZURE_PASSWORD = os.environ["AZURE_SQL_PASSWORD"]

## 3: Read movies + users from Azure SQL

In [3]:
df_movies = spark.read.format("jdbc") \
    .option("url", AZURE_JDBC_URL) \
    .option("dbtable", "movies") \
    .option("user", AZURE_USER) \
    .option("password", AZURE_PASSWORD) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

df_movies.show(5)
print("Movies count:", df_movies.count())

df_users = spark.read.format("jdbc") \
    .option("url", AZURE_JDBC_URL) \
    .option("dbtable", "users") \
    .option("user", AZURE_USER) \
    .option("password", AZURE_PASSWORD) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

df_users.show(5)
print("Users count:", df_users.count())

+--------+--------------------+--------------------+
|movie_id|               title|              genres|
+--------+--------------------+--------------------+
|       1|    Toy Story (1995)|Animation|Childre...|
|       2|      Jumanji (1995)|Adventure|Childre...|
|       3|Grumpier Old Men ...|      Comedy|Romance|
|       4|Waiting to Exhale...|        Comedy|Drama|
|       5|Father of the Bri...|              Comedy|
+--------+--------------------+--------------------+
only showing top 5 rows
Movies count: 3883
+-------+------+---+----------+--------+
|user_id|gender|age|occupation|zip_code|
+-------+------+---+----------+--------+
|      1|     F|  1|        10|   48067|
|      2|     M| 56|        16|   70072|
|      3|     M| 25|        15|   55117|
|      4|     M| 45|         7|   02460|
|      5|     M| 25|        20|   55455|
+-------+------+---+----------+--------+
only showing top 5 rows
Users count: 6040


## 4: Read the Hadoop CSV

In [4]:
df_ratings_raw = spark.read.csv(
    os.path.expanduser("~/bigdata_lab/data/movielens/ratings_raw.csv"),
    header=True, inferSchema=True
)
df_ratings_raw.show(5)
print("Ratings count:", df_ratings_raw.count())

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|    1193|     5|978300760|
|      1|     661|     3|978302109|
|      1|     914|     3|978301968|
|      1|    3408|     4|978300275|
|      1|    2355|     5|978824291|
+-------+--------+------+---------+
only showing top 5 rows
Ratings count: 1000209


## 5: Schema check

In [5]:
df_movies.printSchema()
df_users.printSchema()
df_ratings_raw.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- occupation: integer (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

